# Agentic MDS — start here in Colab

**Do not paste `Agentic_MDS.ipynb` into a code cell.** That file is JSON. Python then hits `NameError: name 'true' is not defined`.

Open this notebook as a notebook:

- [Open Colab_Start_Here.ipynb in Google Colab](https://colab.research.google.com/github/rockyforever8-sys/Agentic-MDS/blob/main/Colab_Start_Here.ipynb)
- Or Colab menu **File → Upload notebook** and pick this `.ipynb` file

Then run the cells below in order. Put secrets in the 🔑 key icon on the left sidebar.

In [ ]:
# Cell 1 — clone the repo and install. Run this in Colab, not by pasting the .ipynb JSON.
import os, pathlib, subprocess, sys

ROOT = pathlib.Path("/content/Agentic-MDS")
if not (ROOT / "imds_agent_v2.py").exists():
    subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/rockyforever8-sys/Agentic-MDS.git", str(ROOT)])
os.chdir(ROOT)
print("Working directory:", os.getcwd())

%pip install -q playwright openpyxl nest_asyncio pyotp cryptography ipywidgets
!playwright install chromium
print("Install done.")

In [ ]:
# Cell 2 — no IMDS login. Confirms the scripts compile.
!python -m py_compile imds_decisions.py imds_secrets.py imds_agent_v2.py
!python imds_agent_v2.py --self-test

In [ ]:
# Cell 3 — one button. Set Colab Secrets first (key icon, left sidebar).
import os
from pathlib import Path
from IPython.display import display
import ipywidgets as widgets

try:
    from google.colab import userdata, drive
    for _key in (
        "IMDS_USERNAME", "IMDS_PASSWORD", "OTP_SECRET", "IMDS_MASTER_KEY",
        "IMDS_CONTACT_NAME", "RECIPIENT_COMPANY_IDS", "NUM_ITERATIONS",
    ):
        try:
            val = userdata.get(_key)
            if val:
                os.environ[_key] = val
        except Exception:
            pass
    if not Path("/content/drive/MyDrive").exists():
        try:
            drive.mount("/content/drive")
        except Exception:
            pass
except ImportError:
    pass

os.environ.setdefault("NUM_ITERATIONS", "10")
os.environ.setdefault("IMDS_AUTO_ACCEPT", "1")
os.environ.setdefault("IMDS_AUTO_REJECT", "1")
os.environ.setdefault("IMDS_AUTO_FORWARD", "1")
os.environ.setdefault("IMDS_HOLD_AMBER", "0")

from imds_secrets import apply_stored_credentials, missing_secret_keys
apply_stored_credentials(persist=True)

run_btn = widgets.Button(
    description="Run IMDS until complete",
    button_style="success",
    layout=widgets.Layout(width="280px", height="48px"),
)
out = widgets.Output()


def _on_run(_):
    with out:
        out.clear_output()
        apply_stored_credentials(persist=True)
        missing = missing_secret_keys()
        if missing:
            raise RuntimeError(
                "Private secrets missing: " + ", ".join(missing) +
                ". Add IMDS_USERNAME, IMDS_PASSWORD, OTP_SECRET, and IMDS_MASTER_KEY in Colab Secrets."
            )
        from imds_agent_v2 import orchestrate
        rc = orchestrate()
        print("exit code", rc)
        report = Path("imds_output/mds_status_report.csv")
        if report.exists():
            try:
                import pandas as pd
                from IPython.display import display as show
                show(pd.read_csv(report))
            except Exception:
                print(report.read_text())


run_btn.on_click(_on_run)
display(run_btn, out)
print("Secrets loaded:", not bool(missing_secret_keys()), "| click the green button")
